In [1]:
import sys

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from time import time
from dataclasses import dataclass
import numpy as np
from typing import Dict, List

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig, EmbedderModel, EmbedderModelConfig, VectorDBConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import Neo4jGraphDriver
from src.utils.data_structs import create_id_for_node_pair, NodeType
from src.qa_pipeline.knowledge_retriever.cache import KeyValueStore, KeyValueStoreConfig

/home/dzigen/Desktop/PersonalAI/Personal-AI/pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
@dataclass
class AStarMetricsConfig:
    h_metric_name: str = 'weight_with_short_path'
    d_metric_name: str = 'ip'

class AStarMetrics(Neo4jGraphDriver):
    def __init__(self, kg_model: KnowledgeGraphModel, accepted_node_types: str, config: AStarMetricsConfig = AStarMetricsConfig(), cache: KeyValueStore = None):
        self.kg_model = kg_model
        self.cache = cache
        self.config = config
        self.accepted_node_types = accepted_node_types
        self.metrics_map = {
            'ip': self.precomputed_dist,
            'constant': lambda v1, v2, U, parent: 1,
            'weight_with_short_path': self.weighted_short_path,
            'avg_weighted_with_short_path': self.avg_weighted_short_path,
        }

    def compute_d_metric(self, *args, **kwargs) -> float:
        return self.metrics_map[self.config.d_metric_name](*args, **kwargs)

    def compute_h_metric(self, *args, **kwargs) -> float:
        return self.metrics_map[self.config.h_metric_name](*args, **kwargs)

    def get_nodes_path(self, parent: Dict[str, str], U: List[str], end_node_id: str) -> List[str]:
        #end_node_id = U[-1] if (end_node_id not in parent) else end_node_id
        path, end_flag, cur_n = [end_node_id], False, end_node_id
        while not end_flag:
            next_n = parent[cur_n]
            if next_n is None:
                end_flag = True
            else:
                path.append(next_n)
                cur_n = next_n

        return path

    def precomputed_dist(self, node1_id: str, node2_id: str, **kwargs) -> float:
        pair_id = create_id_for_node_pair(node1_id, node2_id)
        cache_key = ('test', 'dist', pair_id)
        dist = None
        if self.cache.is_key_exists(cache_key):
            print("exists")
            dist = self.cache.get_value_by_key(cache_key)['v']
        else:
            print("calculating")
            instances = self.kg_model.embeddings_db.vectordbs['nodes'].read([node1_id, node2_id], includes=['embeddings'])
            # calculation ip distance 
            dist = 1 - np.dot(instances[0].embedding, instances[1].embedding)
            self.cache.save_kv_pair(cache_key, {'v': dist})
            
        return dist

    def dijkstra(self, s_node_id, e_node_id):
        # Используемая реализация алгоритма Дейкстры: https://ru.wikibooks.org/wiki/%D0%A0%D0%B5%D0%B0%D0%BB%D0%B8%D0%B7%D0%B0%D1%86%D0%B8%D0%B8_%D0%B0%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC%D0%BE%D0%B2/%D0%90%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC_%D0%94%D0%B5%D0%B9%D0%BA%D1%81%D1%82%D1%80%D1%8B
        available_nodes = {s_node_id: 0}
        parent = {s_node_id: None}
        passed_nodes_counter = 0
        while len(available_nodes) > 0:
            min_weight = 1000001
            ID_min_weight = -1
            for node_id, weight in available_nodes.items():
                if weight < min_weight:
                    min_weight = weight
                    ID_min_weight = node_id
            
            if ID_min_weight == e_node_id:
                print("passed nodes: ", passed_nodes_counter)
                if self.cache is not None:
                    pair_id = create_id_for_node_pair(s_node_id, ID_min_weight)
                    self.cache.save_kv_pair(('test', 'short_path', pair_id), {'v': available_nodes[ID_min_weight]})
                return min_weight

            adjenced_nodes_ids = self.get_adjecent_nodes(ID_min_weight, parent[ID_min_weight], self.accepted_node_types)

            for adj_n_id in adjenced_nodes_ids:
                if (adj_n_id not in available_nodes) or ((available_nodes[ID_min_weight] + 1) < available_nodes[adj_n_id]):
                    available_nodes[adj_n_id] = available_nodes[ID_min_weight] + 1
                    parent[adj_n_id] = ID_min_weight
            
            pair_id = create_id_for_node_pair(s_node_id, ID_min_weight)
            self.cache.save_kv_pair(('test', 'short_path', pair_id), {'v': available_nodes[ID_min_weight]})
            del available_nodes[ID_min_weight]
            passed_nodes_counter += 1

        # между вершинами нет пути
        return 1000001

    def precomputed_short_path(self, node1_id: str, node2_id: str) -> float:
        pair_id = create_id_for_node_pair(node1_id, node2_id)
        cache_key = ('test', 'short_path', pair_id)
        if self.cache.is_key_exists(cache_key):
            print("exists")
            short_path = self.cache.get_value_by_key(cache_key)['v']
        else:
            print("calculating")
            short_path = self.dijkstra(node1_id, node2_id)

        return short_path

    def weighted_short_path(self, node1_id: str, node2_id: str, **kwargs) -> float:
        short_path_len = self.precomputed_short_path(node1_id, node2_id)
        w = self.precomputed_dist(node1_id, node2_id)
        return short_path_len * w

    def avg_weighted_short_path(self, node1_id: str, node2_id: str, U: List[str], parent: Dict[str, str]) -> float:
        nodes_path = self.get_nodes_path(parent, U, node1_id)
        acc_dist = 0
        for i in range(len(nodes_path)-1):
            acc_dist += self.precomputed_dist(nodes_path[i], nodes_path[i+1])
        acc_dist += self.precomputed_dist(node1_id, node2_id)

        short_path_len = self.precomputed_short_path(node1_id, node2_id)
        return np.mean(acc_dist) * short_path_len


In [3]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri='bolt://localhost:7687', user='neo4j', pwd='password', db_name='diaasq2'),
    embeddings_db=EmbeddingsDatabaseConnection(EmbeddingsDatabaseConnectionConfig(
        embedder_config=EmbedderModelConfig(model_name_or_path='../../models/intfloat/multilingual-e5-small'),
        nodes_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_nodes/v10/densedb', 'vectorized_nodes', is_exist=True, need_to_clear=False
        ),
        triplets_db_config=VectorDBConnectionConfig(
            '../../data/graph_structures/vectorized_triplets/v6/densedb', 'vectorized_triplets', is_exist=True, need_to_clear=False
        )
    ))
)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [13]:
cache = KeyValueStore()

In [14]:
astar_metrics = AStarMetrics(kg_model, cache=cache, accepted_node_types=f'["{NodeType.object.value}","{NodeType.hyper.value}","{NodeType.episodic.value}"]')

In [6]:
data = kg_model.embeddings_db.vectordbs['nodes'].collection.get(include=['embeddings'], limit=10)

In [7]:
data['ids']

['4:d6769385-179e-400c-982b-6502bc94d278:0',
 '4:d6769385-179e-400c-982b-6502bc94d278:1',
 '4:d6769385-179e-400c-982b-6502bc94d278:2',
 '4:d6769385-179e-400c-982b-6502bc94d278:3',
 '4:d6769385-179e-400c-982b-6502bc94d278:4',
 '4:d6769385-179e-400c-982b-6502bc94d278:5',
 '4:d6769385-179e-400c-982b-6502bc94d278:6',
 '4:d6769385-179e-400c-982b-6502bc94d278:7',
 '4:d6769385-179e-400c-982b-6502bc94d278:8',
 '4:d6769385-179e-400c-982b-6502bc94d278:9']

In [16]:
s_time = time()
astar_metrics.weighted_short_path('4:d6769385-179e-400c-982b-6502bc94d278:2', '4:d6769385-179e-400c-982b-6502bc94d278:1')
e_time = time()
print(e_time - s_time)

calculating
passed nodes:  22
calculating
0.16015958786010742


In [18]:
s_time = time()
astar_metrics.precomputed_dist('4:d6769385-179e-400c-982b-6502bc94d278:9', '4:d6769385-179e-400c-982b-6502bc94d278:1')
e_time = time()
print(e_time - s_time)

exists
0.0006651878356933594
